# TASK-20 · Native Ollama Python SDK vs LangChain

Native `ollama.Client` has `chat()` and `embed()` calls. It passes generation settings inside `options={...}`. The **models and inputs match** the Local script.

## Setup: hosted accelerator and dependencies

The next cells inspect Colab's optional GPU, install **Python packages**, install required `zstd` for extraction, then install the official **Ollama Linux server**, start its API, and download two small models. On each notebook's fresh runtime the setup repeats. Official installation script executes remote code; only use the official Ollama domain.

In [ ]:
import shutil
import subprocess

print("NVIDIA GPU utility available:", bool(shutil.which("nvidia-smi")))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU assigned — expect CPU inference.")

In [ ]:
%pip install -q 'langchain-ollama>=1.0,<2' 'langchain-core>=1.0,<2' 'ollama>=0.6,<1'

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

def ollama_ready():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_ready():
    server_log = open("/tmp/ollama-server.log", "a")
    server = subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if ollama_ready():
            break
        time.sleep(2)
assert ollama_ready(), "Ollama failed to start; inspect /tmp/ollama-server.log"
print("Ollama API responding on localhost:11434")

In [ ]:
import subprocess
for model in ("qwen3:1.7b", "embeddinggemma-2:270m"):
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"], check=True)

## Native chat

Compare `client.chat()` with the earlier `ChatOllama(...).invoke()` call. `think=False` is the native equivalent of turning off supported thinking behavior.

In [ ]:
from ollama import Client
client = Client(host="http://localhost:11434")
reply = client.chat(
    model="qwen3:1.7b",
    messages=[
        {"role": "system", "content": "You are an AI tutor. Reply concisely."},
        {"role": "user", "content": "Explain RAG in three simple sentences."},
    ],
    stream=False, think=False, keep_alive="10m",
    options={
        "temperature": 0.3, "top_p": 0.9, "top_k": 40,
        "num_ctx": 4096, "num_predict": 128,
        "repeat_penalty": 1.1, "seed": 42,
    },
)
print("Answer:", reply.message.content)
print("Input tokens:", reply.prompt_eval_count)
print("Output tokens:", reply.eval_count)

## Native embeddings

`client.embed(input=[...])` embeds more than one text at once; the returned field is `embeddings`. These vectors are not natural language.

In [ ]:
result = client.embed(
    model="embeddinggemma-2:270m",
    input=[
        "What is retrieval-augmented generation?",
        "RAG retrieves relevant documents to support answers.",
    ],
    keep_alive="10m",
)
print("Vectors:", len(result.embeddings))
print("Vector dimensions:", len(result.embeddings[0]))
assert len(result.embeddings) == 2

## Inspect GPU/CPU execution

`PROCESSOR` reports actual placement for currently loaded models.

In [ ]:
subprocess.run(["ollama", "ps"], check=True)